<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tool definition

In [ ]:
%%bash
apt install -y zstd pciutils lshw
curl -fsSL https://ollama.com/install.sh | sh
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-ollama ollama

In [ ]:
!nohup ollama serve &

In [ ]:
!ollama pull gpt-oss:20b

In [2]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [8]:
from langchain import tools

@tools.tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

square_root.invoke(input={"x": 467})

21.61018278497431

In [9]:
from langchain import tools

@tools.tool(
    name_or_callable="square_root",
    description="Calculate the square root of a number"
)
def tool1(x: float) -> float:
    return x ** 0.5

tool1.invoke(input={"x": 467})

21.61018278497431

## Adding to agents

In [16]:
import langchain_ollama
from langchain import tools, agents, messages
import time

model = langchain_ollama.ChatOllama(
    model="gpt-oss:20b",
    temperature=0.1
)

@tools.tool(
    name_or_callable="square_root",
    description="Calculate the square root of a number"
)
def tool1(x: float) -> float:
    return x ** 0.5

agent = agents.create_agent(
    model=model,
    tools=[tool1],
    system_prompt="""
        You are an arithmetic wizard.
        Use your tools to calculate the square root and square of any number.
    """
)

question = messages.HumanMessage(content="""
    What is the square root of 467?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 3.75s seconds
================================ Human Message =================================


    What is the square root of 467?

================================== Ai Message ==================================
Tool Calls:
  square_root (85e43941-97d9-4f66-a4b9-5cb5aeaff9c5)
 Call ID: 85e43941-97d9-4f66-a4b9-5cb5aeaff9c5
  Args:
    x: 467
================================= Tool Message =================================
Name: square_root

21.61018278497431
================================== Ai Message ==================================

The square root of 467 is approximately **21.61018278497431**.
